# Mutation → Mechanism → Therapy — Main Run Notebook

**Repo:** [github.com/lightflow16/mutation-mechanism-therapy-amd](https://github.com/lightflow16/mutation-mechanism-therapy-amd)

**Clone once** — run the Python clone cell below (section 0). Do **not** paste `git clone ...` as bare Python; that causes `SyntaxError`.

AMD notebook: start in `/workspace`. Colab: start in `/content`.

## 0 — Clone + confirm repo root

In [ ]:
from pathlib import Path
import os, subprocess

ROOT = Path(".").resolve()
repo_dir = ROOT / "mutation-mechanism-therapy-amd"

if (ROOT / "src" / "pipeline.py").exists():
    print("Already in repo root:", ROOT)
elif (repo_dir / "src" / "pipeline.py").exists():
    os.chdir(repo_dir)
    print("Entered clone:", Path(".").resolve())
else:
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/lightflow16/mutation-mechanism-therapy-amd.git",
        ],
        check=True,
        cwd=ROOT,
    )
    os.chdir(repo_dir)
    print("Cloned to:", Path(".").resolve())

In [ ]:
# Clone once in terminal (not here), then open this notebook from the repo root:
#   cd /workspace && git clone --depth 1 https://github.com/lightflow16/mutation-mechanism-therapy-amd.git
#   cd mutation-mechanism-therapy-amd

from pathlib import Path
ROOT = Path('.').resolve()
assert (ROOT / 'src' / 'pipeline.py').exists(), (
    'Run from repo root. Clone: git clone https://github.com/lightflow16/mutation-mechanism-therapy-amd.git'
)
print('ROOT =', ROOT)

## 1 — Environment + paths (CPU ok, no GPU yet)

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('.').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import configure_paths, is_rocm, setup_env
from src.serving import verify_gpu_torch

configure_paths()
setup_env()

import torch
print('torch', torch.__version__)
print('platform:', 'ROCm/AMD' if is_rocm() else 'CUDA (Colab) or local')
print('LLM_BACKEND', os.environ.get('LLM_BACKEND', 'auto'))
print('GPU verify (CPU ok if not attached yet):', verify_gpu_torch())

In [ ]:
!pip install -q -r requirements.txt openai

In [ ]:
!bash scripts/setup_external.sh

## 2 — Instant demo (cached traces, NO GPU, NO vLLM)

In [ ]:
from src.pipeline import run_case
import json

for gene, mut in [('EGFR','L858R'), ('PIK3CA','E545K'), ('TP53','R175H')]:
    r = run_case(gene, mut, architecture='blackboard', use_cached_trace=True, live_evidence=False)
    tr = r['reasoning'].get('target_reasoning', r['reasoning'])
    print(gene, mut, '->', r['route'], '| therapies:', tr.get('therapy',{}).get('sensitivity'))

## 3 — Attach GPU ONLY from here onward

**AMD:** use transformers (default). **Do NOT** `pip install vllm` — it breaks ROCm torch.

**Colab:** transformers works out of the box. Optional vLLM: `bash scripts/install_vllm_colab.sh` then `bash scripts/start_vllm.sh`.

Run cells below back-to-back; detach GPU when idle on AMD.

In [ ]:
import time
GPU_ATTACH_T0 = time.time()  # budget proxy (~240 min)
print('GPU attached at', GPU_ATTACH_T0)

In [ ]:
from src.config import is_rocm
from src.serving import platform_summary, verify_gpu_torch
from src.llm_client import use_vllm

print(platform_summary())
gpu = verify_gpu_torch()
assert gpu["ok"], f"GPU broken: {gpu.get('error')} — restart session if you ran pip install vllm on AMD"
print("LLM routing:", "vllm" if use_vllm() else "transformers")

In [ ]:
# OPTIONAL: vLLM on Colab CUDA only (skip on AMD)
from src.config import is_rocm

if is_rocm():
    print("Skipping vLLM on ROCm — using transformers (LLM_BACKEND=transformers)")
else:
    print("Optional Colab vLLM — uncomment to install cu128 wheel:")
    print("  !bash scripts/install_vllm_colab.sh")
    print("  !bash scripts/start_vllm.sh")
    # !bash scripts/install_vllm_colab.sh
    # !bash scripts/start_vllm.sh

In [ ]:
# Live single-agent (transformers + GPU) — first run downloads ~15 GB
from src.pipeline import run_case
r = run_case('EGFR', 'L858R', architecture='single', use_cached_trace=False)
print(r['reasoning'])

In [ ]:
# Live blackboard via transformers (slow) OR use cached traces for demo
# r = run_case('EGFR','L858R', architecture='blackboard', use_cached_trace=False)
# print(r['reasoning'].get('target_reasoning', r['reasoning']))

In [ ]:
# LoRA training (~10-30 min GPU)
!python train/lora_sft.py

In [ ]:
# TP53 rescue branch (ThermoMPNN + ESMFold — GPU)
from src.pipeline import run_case
r = run_case('TP53', 'R175H', architecture='blackboard', use_cached_trace=False, run_rescue_branch=True)
print(r.get('rescue', {}).keys())

In [ ]:
# Gradio demo (can run on CPU with cached traces)
# !python app.py

## 4 — Download submission artifacts

In [ ]:
!tar -czf /workspace/shared/submission_bundle.tgz \
  metrics/ data/traces/ deck/ \
  2>/dev/null || tar -czf /workspace/shared/submission_bundle.tgz -C /workspace/shared metrics data/traces deck 2>/dev/null || echo 'Adjust paths if needed'
!ls -lh /workspace/shared/*.tgz /workspace/shared/metrics/ 2>/dev/null | head